# Lesson 7.4: Color Image Smoothing and Sharpening
## Biomedical Image Processing - Color Image Processing

### Topics:
- Applying spatial filters to color images
- Per-channel filtering
- Color image smoothing (averaging, Gaussian)
- Color image sharpening (Laplacian, unsharp masking)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve

# Create a 200x200 color image with distinct colored regions
image = np.zeros((200, 200, 3), dtype=np.float64)

# Red region (top-left)
image[0:100, 0:100] = [1.0, 0.0, 0.0]
# Green region (top-right)
image[0:100, 100:200] = [0.0, 1.0, 0.0]
# Blue region (bottom-left)
image[100:200, 0:100] = [0.0, 0.0, 1.0]
# Yellow region (bottom-right)
image[100:200, 100:200] = [1.0, 1.0, 0.0]

# Add a white circle in the center
y, x = np.ogrid[-100:100, -100:100]
mask = x**2 + y**2 < 40**2
image[mask] = [1.0, 1.0, 1.0]

# Add Gaussian noise
np.random.seed(42)
noise = np.random.normal(0, 0.1, image.shape)
noisy_image = np.clip(image + noise, 0, 1)

# Display original and noisy images
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(image)
axes[0].set_title('Original Image')
axes[0].axis('off')
axes[1].imshow(noisy_image)
axes[1].set_title('Noisy Image (σ=0.1)')
axes[1].axis('off')
plt.tight_layout()
plt.show()

## 1. Color Image Smoothing

Smoothing (low-pass filtering) reduces noise by averaging neighboring pixels.

For color images, we apply the filter **independently to each channel**:

$$g_c(x,y) = \sum_{s=-a}^{a} \sum_{t=-b}^{b} w(s,t) \cdot f_c(x+s, y+t), \quad c \in \{R, G, B\}$$

### Common smoothing filters:
- **Averaging filter:** equal weights (box filter)
- **Gaussian filter:** weights follow a Gaussian distribution — more weight to center pixels

The Gaussian kernel with standard deviation $\sigma$:

$$G(x,y) = \frac{1}{2\pi\sigma^2} e^{-\frac{x^2+y^2}{2\sigma^2}}$$

In [ ]:
def create_gaussian_kernel(size, sigma):
    """Create a 2D Gaussian kernel."""
    ax = np.arange(size) - size // 2
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2.0 * sigma**2))
    return kernel / kernel.sum()

# Averaging filter (box filter) - 5x5
avg_kernel = np.ones((5, 5)) / 25.0

# Gaussian filter - 5x5, sigma=1.0
gauss_kernel = create_gaussian_kernel(5, sigma=1.0)

# Apply filters to each channel independently
def apply_filter_color(img, kernel):
    """Apply a 2D filter to each channel of a color image."""
    result = np.zeros_like(img)
    for ch in range(3):
        result[:, :, ch] = convolve(img[:, :, ch], kernel, mode='reflect')
    return np.clip(result, 0, 1)

smoothed_avg = apply_filter_color(noisy_image, avg_kernel)
smoothed_gauss = apply_filter_color(noisy_image, gauss_kernel)

# Larger Gaussian for stronger smoothing
gauss_kernel_large = create_gaussian_kernel(11, sigma=3.0)
smoothed_gauss_large = apply_filter_color(noisy_image, gauss_kernel_large)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(noisy_image)
axes[0, 0].set_title('Noisy Image')

axes[0, 1].imshow(smoothed_avg)
axes[0, 1].set_title('Averaging Filter (5×5)')

axes[1, 0].imshow(smoothed_gauss)
axes[1, 0].set_title('Gaussian Filter (5×5, σ=1.0)')

axes[1, 1].imshow(smoothed_gauss_large)
axes[1, 1].set_title('Gaussian Filter (11×11, σ=3.0)')

for ax in axes.flat:
    ax.axis('off')

plt.suptitle("Color Image Smoothing", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Averaging filter blurs uniformly — edges become very soft.")
print("Gaussian filter provides smoother blurring with less edge artifacts.")
print("Larger kernel / higher sigma = stronger smoothing but more detail loss.")

---

## 2. Color Image Sharpening

Sharpening enhances edges by amplifying high-frequency components.

### 2.1 Laplacian Sharpening

The **Laplacian** operator detects edges by computing the second derivative:

$$\nabla^2 f = \frac{\partial^2 f}{\partial x^2} + \frac{\partial^2 f}{\partial y^2}$$

Common Laplacian kernel:

$$\begin{bmatrix} 0 & -1 & 0 \\ -1 & 4 & -1 \\ 0 & -1 & 0 \end{bmatrix}$$

Sharpened image: $g(x,y) = f(x,y) + c \cdot \nabla^2 f(x,y)$

### 2.2 Unsharp Masking

1. Blur the image to get $f_{smooth}$
2. Compute the mask: $\text{mask} = f - f_{smooth}$
3. Add the mask back: $g = f + k \cdot \text{mask}$

where $k$ controls the sharpening strength.

In [ ]:
# Laplacian sharpening on color image
laplacian_kernel = np.array([
    [ 0, -1,  0],
    [-1,  4, -1],
    [ 0, -1,  0]
], dtype=np.float64)

# Apply Laplacian to each channel of the original (clean) image
laplacian_edges = apply_filter_color(image, laplacian_kernel)

# Sharpened = original + c * Laplacian
c_sharp = 1.0
sharpened_lap = np.clip(image + c_sharp * laplacian_edges, 0, 1)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(image)
axes[0].set_title("Original")

# Scale Laplacian edges for visibility
lap_vis = np.clip(laplacian_edges * 5 + 0.5, 0, 1)
axes[1].imshow(lap_vis)
axes[1].set_title("Laplacian Edges (scaled)")

axes[2].imshow(sharpened_lap)
axes[2].set_title("Laplacian Sharpened (c=1.0)")

for ax in axes:
    ax.axis('off')

plt.suptitle("Laplacian Sharpening", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Unsharp masking on color image
gauss_kernel_unsharp = create_gaussian_kernel(7, sigma=2.0)

# Step 1: Blur the image
blurred = apply_filter_color(image, gauss_kernel_unsharp)

# Step 2: Compute the unsharp mask
mask = image - blurred

# Step 3: Add the mask back with different strengths
k_values = [1.0, 2.0, 3.0]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

axes[0].imshow(image)
axes[0].set_title("Original")

for idx, k in enumerate(k_values):
    sharpened = np.clip(image + k * mask, 0, 1)
    axes[idx + 1].imshow(sharpened)
    axes[idx + 1].set_title(f"Unsharp Mask (k={k})")

for ax in axes:
    ax.axis('off')

plt.suptitle("Unsharp Masking with Different Strengths", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("Higher k values produce stronger sharpening but may introduce artifacts.")
print("Unsharp masking is widely used in medical imaging to enhance tissue boundaries.")

---

## 3. Smoothing Then Sharpening: Denoise + Enhance Pipeline

A common pipeline in biomedical imaging:
1. **Smooth** to remove noise
2. **Sharpen** to recover edges lost during smoothing

This combination gives cleaner results than either operation alone.

In [ ]:
# Pipeline: Smooth noisy image, then sharpen
# Step 1: Gaussian smoothing to remove noise
denoised = apply_filter_color(noisy_image, gauss_kernel)

# Step 2: Unsharp masking to recover edges
blurred_denoised = apply_filter_color(denoised, gauss_kernel_unsharp)
mask_denoised = denoised - blurred_denoised
sharpened_denoised = np.clip(denoised + 2.0 * mask_denoised, 0, 1)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

axes[0, 0].imshow(image)
axes[0, 0].set_title("Original (Clean)")

axes[0, 1].imshow(noisy_image)
axes[0, 1].set_title("Noisy Image")

axes[1, 0].imshow(denoised)
axes[1, 0].set_title("Step 1: Gaussian Smoothed")

axes[1, 1].imshow(sharpened_denoised)
axes[1, 1].set_title("Step 2: Smoothed + Sharpened")

for ax in axes.flat:
    ax.axis('off')

plt.suptitle("Denoise + Enhance Pipeline", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("The pipeline reduces noise while preserving edges better than smoothing alone.")

## Summary

What we learned:
1. **Per-channel filtering** = apply spatial filters independently to R, G, B channels
2. **Averaging filter** = simple smoothing with equal weights; blurs edges significantly
3. **Gaussian filter** = weighted smoothing with better edge preservation than averaging
4. **Laplacian sharpening** = second derivative operator enhances edges; $g = f + c \cdot \nabla^2 f$
5. **Unsharp masking** = subtract blurred version to get detail mask, then add it back with strength $k$
6. **Denoise + Sharpen pipeline** = first smooth to remove noise, then sharpen to recover edges